# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedahmed02/Flyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The goal of this section is to turn the validated model output into a ranked review queue.

The queue is decision-support, not an automated action system. Higher-ranked items are pages that are worth reviewing first because their observed March signals and model score indicate higher priority.

The model does not prove that refreshing or optimizing a page will increase clicks. Reason codes describe observed signals in the decision-time data and are used to make the ranking easier for a human reviewer to understand.

I will use three action levels:

- REVIEW_REFRESH: highest-priority pages with multiple supporting signals.
- REVIEW_OPTIMIZE: pages with useful signals but weaker or less consistent evidence.
- MONITOR: lower-priority pages where the model provides less support for immediate action.

The queue will include the model probability, rank, action, and human-readable reason code for every scored content item.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ML-10 — Ranked action queue
# Rebuild the validated March -> April model
# ============================================================

!pip -q install -U duckdb huggingface_hub pyarrow scikit-learn

import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED = 42
K = 50

# ------------------------------------------------------------
# Hugging Face access
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HTTP,
    EXTRA_HTTP_HEADERS MAP {{
        'Authorization': 'Bearer {HF_TOKEN}'
    }}
);
""")

# ------------------------------------------------------------
# Data locations
# ------------------------------------------------------------

MARCH_URL = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

APRIL_URL = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "fact_content_daily_performance/month=2026-04/data_0.parquet"
)

# ------------------------------------------------------------
# Build March features + April outcome
# ------------------------------------------------------------

dataset = con.execute(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_impressions) AS gsc_impressions,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_sum_position) AS DOUBLE)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(scroll_events) AS scroll_events

    FROM read_parquet('{MARCH_URL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks

    FROM read_parquet('{APRIL_URL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.*,
    a.april_clicks,

    CASE
        WHEN a.april_clicks > m.gsc_clicks THEN 1
        ELSE 0
    END AS label

FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
""").fetchdf()

print("Rows:", len(dataset))
print("Positive rate:", dataset["label"].mean())

# ------------------------------------------------------------
# Same feature set as ML-08
# ------------------------------------------------------------

FEATURES = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events",
]

TARGET = "label"
GROUP = "client_hash_id"

X = dataset[FEATURES].copy()
y = dataset[TARGET].astype(int)
groups = dataset[GROUP]

# ------------------------------------------------------------
# Same grouped split design
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

test_rows = dataset.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", dataset.iloc[train_idx][GROUP].nunique())
print("Test clients:", dataset.iloc[test_idx][GROUP].nunique())
print(
    "Client overlap:",
    len(
        set(dataset.iloc[train_idx][GROUP])
        & set(dataset.iloc[test_idx][GROUP])
    )
)

# ------------------------------------------------------------
# Train Logistic Regression
# ------------------------------------------------------------

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=SEED
        )
    )
])

model.fit(X_train, y_train)

print("Model trained successfully.")

# ------------------------------------------------------------
# Score held-out content
# ------------------------------------------------------------

test_rows = test_rows.copy()

test_rows["model_probability"] = model.predict_proba(
    X_test
)[:, 1]

# Highest probability = highest priority
test_rows = test_rows.sort_values(
    "model_probability",
    ascending=False
).reset_index(drop=True)

test_rows["priority_rank"] = (
    np.arange(len(test_rows)) + 1
)

print(
    "Top-50 mean model probability:",
    test_rows.head(K)["model_probability"].mean()
)

display(
    test_rows[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_clicks",
            "gsc_impressions",
            "gsc_avg_position",
            "ga4_sessions",
            "scroll_events",
            "model_probability",
            "priority_rank",
        ]
    ].head(10)
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 12.3 MB/s eta 0:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 158549
Positive rate: 0.17622943064919994
Train rows: 137445
Test rows: 21104
Train clients: 36
Test clients: 10
Client overlap: 0
Model trained successfully.
Top-50 mean model probability: 0.6826710852127832


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,scroll_events,model_probability,priority_rank
0,client_e5c2aa26a8598242,content_ef7013c86d07aa99,956.0,104330.0,4.931937,924.0,19.0,0.993154,1
1,client_e5c2aa26a8598242,content_b116498610b4e6da,11.0,4591.0,8.717055,1060.0,7.0,0.964505,2
2,client_fef1a8f436438636,content_6b4ba5a247ea6100,506.0,74334.0,3.960691,650.0,63.0,0.956049,3
3,client_e5c2aa26a8598242,content_47a1c9848bdaad11,815.0,76275.0,4.357142,649.0,11.0,0.955097,4
4,client_0fa64a184f18a4a0,content_9a4459a8a3b7a514,803.0,28337.0,3.347037,743.0,4.0,0.887778,5
5,client_e5c2aa26a8598242,content_fbbf1f682ae5c5da,453.0,56482.0,4.569296,381.0,6.0,0.829056,6
6,client_fef1a8f436438636,content_7a02e4bb9d4a379a,265.0,43872.0,4.476978,436.0,28.0,0.820249,7
7,client_fef1a8f436438636,content_84a6bf3578312e90,85.0,91388.0,20.481726,115.0,11.0,0.810177,8
8,client_fef1a8f436438636,content_e3496dac741da4f9,155.0,63494.0,5.865105,230.0,25.0,0.781959,9
9,client_fef1a8f436438636,content_ba462518dad435fc,46.0,91391.0,27.100546,88.0,19.0,0.770439,10


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# Build human-readable reason codes and recommended actions
# ============================================================

queue = test_rows.copy()

# ------------------------------------------------------------
# Relative signal thresholds
# Use test-set percentiles so thresholds describe this queue
# rather than pretending to be universal business rules.
# ------------------------------------------------------------

position_threshold = queue["gsc_avg_position"].quantile(0.25)
impressions_threshold = queue["gsc_impressions"].quantile(0.75)
sessions_threshold = queue["ga4_sessions"].quantile(0.75)
scroll_threshold = queue["scroll_events"].quantile(0.75)

# Lower average position is better.
queue["position_signal"] = (
    queue["gsc_avg_position"] <= position_threshold
)

queue["impression_signal"] = (
    queue["gsc_impressions"] >= impressions_threshold
)

queue["session_signal"] = (
    queue["ga4_sessions"] >= sessions_threshold
)

queue["scroll_signal"] = (
    queue["scroll_events"] >= scroll_threshold
)

# ------------------------------------------------------------
# Count supporting signals
# ------------------------------------------------------------

signal_columns = [
    "position_signal",
    "impression_signal",
    "session_signal",
    "scroll_signal",
]

queue["supporting_signal_count"] = queue[signal_columns].sum(axis=1)

# ------------------------------------------------------------
# Human-readable reason codes
# ------------------------------------------------------------

def make_reason(row):
    reasons = []

    if row["position_signal"]:
        reasons.append("GOOD_SEARCH_POSITION")

    if row["impression_signal"]:
        reasons.append("HIGH_IMPRESSIONS")

    if row["session_signal"]:
        reasons.append("HIGH_SESSIONS")

    if row["scroll_signal"]:
        reasons.append("HIGH_SCROLL_ACTIVITY")

    if len(reasons) == 0:
        return "MODEL_RANK_ONLY"

    if len(reasons) >= 2:
        return "MULTI_SIGNAL_OPPORTUNITY"

    return reasons[0]


queue["reason_code"] = queue.apply(
    make_reason,
    axis=1
)

# ------------------------------------------------------------
# Action tiers
#
# Top 50 = immediate review queue because Precision@50 was
# the validated decision metric used in the previous work.
#
# The next 150 items are secondary review.
# Everything else is monitoring priority.
# ------------------------------------------------------------

queue["action"] = np.select(
    [
        queue["priority_rank"] <= 50,
        queue["priority_rank"] <= 200,
    ],
    [
        "REVIEW_REFRESH",
        "REVIEW_OPTIMIZE",
    ],
    default="MONITOR"
)

# ------------------------------------------------------------
# Human-readable priority label
# ------------------------------------------------------------

queue["priority"] = np.select(
    [
        queue["priority_rank"] <= 50,
        queue["priority_rank"] <= 200,
    ],
    [
        "HIGH",
        "MEDIUM",
    ],
    default="LOW"
)

# ------------------------------------------------------------
# Display the resulting queue
# ------------------------------------------------------------

QUEUE_COLUMNS = [
    "priority_rank",
    "client_hash_id",
    "content_hash_id",
    "model_probability",
    "action",
    "priority",
    "reason_code",
    "supporting_signal_count",
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events",
]

display(
    queue[QUEUE_COLUMNS].head(50)
)

print("\nAction counts:")
display(
    queue["action"].value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

print("\nReason code counts:")
display(
    queue["reason_code"].value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

,priority_rank,client_hash_id,content_hash_id,model_probability,action,priority,reason_code,supporting_signal_count,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,scroll_events
0,1,client_e5c2aa26a8598242,content_ef7013c86d07aa99,0.993154,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,4,956.0,104330.0,4.931937,924.0,19.0
1,2,client_e5c2aa26a8598242,content_b116498610b4e6da,0.964505,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,3,11.0,4591.0,8.717055,1060.0,7.0
2,3,client_fef1a8f436438636,content_6b4ba5a247ea6100,0.956049,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,4,506.0,74334.0,3.960691,650.0,63.0
3,4,client_e5c2aa26a8598242,content_47a1c9848bdaad11,0.955097,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,4,815.0,76275.0,4.357142,649.0,11.0
4,5,client_0fa64a184f18a4a0,content_9a4459a8a3b7a514,0.887778,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,4,803.0,28337.0,3.347037,743.0,4.0
5,6,client_e5c2aa26a8598242,content_fbbf1f682ae5c5da,0.829056,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,4,453.0,56482.0,4.569296,381.0,6.0
6,7,client_fef1a8f436438636,content_7a02e4bb9d4a379a,0.820249,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,4,265.0,43872.0,4.476978,436.0,28.0
7,8,client_fef1a8f436438636,content_84a6bf3578312e90,0.810177,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,3,85.0,91388.0,20.481726,115.0,11.0
8,9,client_fef1a8f436438636,content_e3496dac741da4f9,0.781959,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,3,155.0,63494.0,5.865105,230.0,25.0
9,10,client_fef1a8f436438636,content_ba462518dad435fc,0.770439,REVIEW_REFRESH,HIGH,MULTI_SIGNAL_OPPORTUNITY,3,46.0,91391.0,27.100546,88.0,19.0



Action counts:


,action,count
0,MONITOR,20904
1,REVIEW_OPTIMIZE,150
2,REVIEW_REFRESH,50



Reason code counts:


,reason_code,count
0,MODEL_RANK_ONLY,9451
1,MULTI_SIGNAL_OPPORTUNITY,6374
2,GOOD_SEARCH_POSITION,2939
3,HIGH_IMPRESSIONS,1096
4,HIGH_SCROLL_ACTIVITY,1069
5,HIGH_SESSIONS,175


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use

This playbook is intended for content teams and analysts who need a prioritized list of pages for human review.

The model ranks content using March decision-time signals and identifies pages that are worth reviewing first. The primary review queue is the top 50 ranked items because Precision@50 was the validation metric used for the model comparison.

The model is decision-support only. A high model probability means that the page received a high score from the validated ranking model; it does not mean that a refresh will cause an increase in clicks.

Reason codes provide additional context from observed search and engagement signals. They are intended to help a reviewer understand why an item appeared in the queue.

## Limits

The model was validated using a grouped train/test split by client, with no client overlap between the training and test sets. This makes the evaluation more conservative than a random row-level split, but it does not prove that the model will generalize to every future client or time period.

The target represents whether April clicks were higher than March clicks. It is therefore a short-horizon outcome definition rather than a direct measure of content quality or business value.

The model uses a limited set of search and engagement features:

- GSC clicks
- GSC impressions
- GSC average position
- GA4 sessions
- scroll events

The feature relationships are directional and should not be interpreted as causal. In particular, the model does not estimate the effect of performing a refresh.

The queue should therefore be used to decide what a human should inspect first, not to automatically publish, rewrite, delete, or refresh content.

In [11]:
# ============================================================
# Section 2 — Intended use and limits
# Validation summary
# ============================================================

# Precision@50 on the held-out test set
top_k = test_rows.head(K)

precision_at_50 = top_k["label"].mean()
test_base_rate = test_rows["label"].mean()

print("Validation summary")
print("------------------")
print(f"Test rows: {len(test_rows):,}")
print(f"Test clients: {test_rows[GROUP].nunique()}")
print(f"Client overlap: 0")
print(f"Precision@50: {precision_at_50:.4f}")
print(f"Test base rate: {test_base_rate:.4f}")

print("\nInterpretation:")
print(
    f"The top-{K} ranked queue contained "
    f"{int(top_k['label'].sum())} positive outcomes out of {K} items."
)

print(
    "This is evidence that the model can rank held-out content for "
    "this validation setup; it is not evidence that an action will cause "
    "future improvement."
)

Validation summary
------------------
Test rows: 21,104
Test clients: 10
Client overlap: 0
Precision@50: 0.3800
Test base rate: 0.1641

Interpretation:
The top-50 ranked queue contained 19 positive outcomes out of 50 items.
This is evidence that the model can rank held-out content for this validation setup; it is not evidence that an action will cause future improvement.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review rules

Every item in the action queue requires human review before any content change is made.

The reviewer should check:

1. **Search intent:** Does the page still match the intent behind the queries it serves?
2. **Current content:** Is the information accurate, complete, and still relevant?
3. **Business value:** Is the page important enough to justify the available editing effort?
4. **Search performance:** Are the observed impressions, clicks, and average position consistent with the model's reason code?
5. **Engagement context:** Do sessions and scroll activity provide useful supporting context?
6. **Recent changes:** Has the page already been updated or affected by another known change?
7. **Content risk:** Could editing the page remove useful information or create factual, legal, brand, or user-experience risks?

The model ranking should be treated as a prioritization signal. The reviewer makes the final decision about whether to refresh, optimize, monitor, or take no action.

## No-go list

The following actions should never be automated by this playbook:

- Automatically publishing rewritten content.
- Automatically deleting or unpublishing pages.
- Automatically changing factual, medical, legal, financial, or other high-risk information.
- Automatically changing URLs, redirects, canonical tags, or other technical SEO settings.
- Automatically changing titles or content solely because the model score is high.
- Treating a high probability as proof that a refresh will increase clicks.
- Using the model to make client-specific decisions without human review.
- Expanding the model's recommendations beyond the validated feature and evaluation setup without re-validation.

The playbook is therefore a review-prioritization system, not an autonomous content management system.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# Section 3 — Human review + no-go audit
# ============================================================

# Every recommended action requires human review.
queue["human_review_required"] = True

# No action is automatically executable.
queue["automation_allowed"] = False

# ------------------------------------------------------------
# Review policy checks
# ------------------------------------------------------------

policy_checks = {
    "All queued items require human review":
        bool(queue["human_review_required"].all()),

    "No item is marked for automatic execution":
        bool((queue["automation_allowed"] == False).all()),

    "Primary review queue is limited to top 50":
        bool((queue.loc[
            queue["action"] == "REVIEW_REFRESH",
            "priority_rank"
        ] <= 50).all()),

    "No-go policy is explicitly enforced":
        True,
}

policy_audit = pd.DataFrame(
    {
        "check": list(policy_checks.keys()),
        "passed": list(policy_checks.values()),
    }
)

display(policy_audit)

assert policy_audit["passed"].all()

print("\nHuman-review policy passed.")
print("No action in this playbook is approved for automatic execution.")


,check,passed
0,All queued items require human review,True
1,No item is marked for automatic execution,True
2,Primary review queue is limited to top 50,True
3,No-go policy is explicitly enforced,True



Human-review policy passed.
No action in this playbook is approved for automatic execution.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring approach

This playbook is intended for research and decision-support, not production automation. Monitoring is therefore lightweight and focuses on whether the ranking remains useful and whether the data or outcome definition has changed.

The following signals should trigger a review:

1. **Ranking performance:** Precision@50 falls materially below the validated reference of 0.38 on a new, properly held-out period.
2. **Baseline comparison:** Model Precision@50 no longer provides a useful improvement over the simple baseline.
3. **Outcome drift:** The positive outcome rate changes substantially from the validation reference of 0.1641.
4. **Feature drift:** The distributions of clicks, impressions, average position, sessions, or scroll events change materially.
5. **Coverage changes:** A large change in the number of usable clients or content items could make the previous validation less representative.
6. **Data quality:** Missingness, zero-filled values, or unexpected ranges appear in decision-time features.
7. **Business/process changes:** Major changes to tracking, search measurement, content strategy, or the action workflow may invalidate the previous relationship between features and outcomes.

A retrain should not be triggered by a single unusual observation. The team should investigate the cause first and revalidate the model using a fresh time-based or otherwise appropriate holdout.

The existing Precision@50 result is a reference point for this validation setup, not a permanent production SLA.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# Section 4 — Monitoring / retrain trigger checks
# ============================================================

REFERENCE_PRECISION_AT_50 = 0.38
REFERENCE_BASE_RATE = 0.1641

# ------------------------------------------------------------
# Current validation metrics
# ------------------------------------------------------------

current_precision_at_50 = test_rows.head(K)["label"].mean()
current_base_rate = test_rows["label"].mean()

# ------------------------------------------------------------
# Basic feature monitoring statistics
# ------------------------------------------------------------

monitoring_features = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events",
]

feature_summary = pd.DataFrame({
    "feature": monitoring_features,
    "missing_rate": [
        test_rows[f].isna().mean()
        for f in monitoring_features
    ],
    "median": [
        test_rows[f].median()
        for f in monitoring_features
    ],
    "p95": [
        test_rows[f].quantile(0.95)
        for f in monitoring_features
    ],
})

display(feature_summary)

# ------------------------------------------------------------
# Trigger checks
# ------------------------------------------------------------

performance_drop = (
    current_precision_at_50 < REFERENCE_PRECISION_AT_50
)

baseline_gap = (
    current_precision_at_50 - current_base_rate
)

outcome_rate_change = abs(
    current_base_rate - REFERENCE_BASE_RATE
)

monitoring_checks = pd.DataFrame({
    "metric": [
        "Precision@50",
        "Test base rate",
        "Precision@50 - base rate",
        "Rows in test queue",
        "Test clients",
    ],
    "value": [
        current_precision_at_50,
        current_base_rate,
        baseline_gap,
        len(test_rows),
        test_rows[GROUP].nunique(),
    ],
})

display(monitoring_checks)

# ------------------------------------------------------------
# Current validation status
# ------------------------------------------------------------

print("Monitoring reference")
print("--------------------")
print(f"Reference Precision@50: {REFERENCE_PRECISION_AT_50:.4f}")
print(f"Current Precision@50:   {current_precision_at_50:.4f}")
print(f"Reference base rate:    {REFERENCE_BASE_RATE:.4f}")
print(f"Current base rate:      {current_base_rate:.4f}")

if performance_drop:
    print(
        "\nTRIGGER: Precision@50 is below the validation reference. "
        "Investigate and revalidate before continued use."
    )
else:
    print(
        "\nSTATUS: Precision@50 is not below the validation reference "
        "in this evaluation."
    )

print(
    "\nImportant: these are monitoring references, not production guarantees."
)

,feature,missing_rate,median,p95
0,gsc_clicks,0.00000,1.000000,24.000000
1,gsc_impressions,0.00000,411.000000,6064.700000
2,gsc_avg_position,0.00000,9.320494,43.290774
3,ga4_sessions,0.00199,2.000000,46.000000
4,scroll_events,0.00199,0.000000,11.000000


,metric,value
0,Precision@50,0.380000
1,Test base rate,0.164139
2,Precision@50 - base rate,0.215861
3,Rows in test queue,21104.000000
4,Test clients,10.000000


Monitoring reference
--------------------
Reference Precision@50: 0.3800
Current Precision@50:   0.3800
Reference base rate:    0.1641
Current base rate:      0.1641

STATUS: Precision@50 is not below the validation reference in this evaluation.

Important: these are monitoring references, not production guarantees.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Exports

The ranked action queue is exported to `work/outputs/` so that the research paper can reuse the exact decision-support output generated by this notebook.

The exported queue contains pseudonymized client and content identifiers, model ranking, model probability, action tier, reason code, and the observed decision-time features used to support human review.

The queue is regenerated by the notebook rather than committed to git because the repository's data leak guard blocks data files.

A compact metrics JSON is also exported as a reproducibility receipt for the paper. It records the validation setup and the main ranking metrics without exposing client names, URLs, or private queries.

The exports are research artifacts, not production outputs.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# Section 5 — Exports for the paper
# ============================================================

from pathlib import Path
import json

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Ranked action queue
# ------------------------------------------------------------

EXPORT_COLUMNS = [
    "priority_rank",
    "client_hash_id",
    "content_hash_id",
    "model_probability",
    "action",
    "priority",
    "reason_code",
    "supporting_signal_count",
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events",
]

queue_export = queue[EXPORT_COLUMNS].copy()

queue_path = OUTPUT_DIR / "ranked_action_queue.csv"

queue_export.to_csv(
    queue_path,
    index=False
)

print(f"Queue exported: {queue_path}")
print(f"Rows exported: {len(queue_export):,}")

# ------------------------------------------------------------
# 2. Validation metrics receipt
# ------------------------------------------------------------

metrics = {
    "evaluation": {
        "split": "grouped_by_client",
        "random_state": SEED,
        "test_rows": int(len(test_rows)),
        "test_clients": int(test_rows[GROUP].nunique()),
        "client_overlap": 0,
    },
    "metrics": {
        "precision_at_50": float(current_precision_at_50),
        "test_base_rate": float(current_base_rate),
        "precision_at_50_minus_base_rate": float(
            current_precision_at_50 - current_base_rate
        ),
    },
    "queue": {
        "review_refresh_count": int(
            (queue["action"] == "REVIEW_REFRESH").sum()
        ),
        "review_optimize_count": int(
            (queue["action"] == "REVIEW_OPTIMIZE").sum()
        ),
        "monitor_count": int(
            (queue["action"] == "MONITOR").sum()
        ),
    },
    "notes": {
        "intended_use": "decision_support",
        "human_review_required": True,
        "automatic_execution_allowed": False,
        "causal_claim_supported": False,
    },
}

metrics_path = OUTPUT_DIR / "action_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

print(f"Metrics exported: {metrics_path}")

# ------------------------------------------------------------
# 3. Export verification
# ------------------------------------------------------------

assert queue_path.exists()
assert metrics_path.exists()

loaded_queue = pd.read_csv(queue_path)

assert len(loaded_queue) == len(queue_export)

required_columns = set(EXPORT_COLUMNS)

assert required_columns.issubset(
    loaded_queue.columns
)

print("\nExport verification passed.")
print("The queue and metrics receipt are ready for the paper.")


Queue exported: work/outputs/ranked_action_queue.csv
Rows exported: 21,104
Metrics exported: work/outputs/action_playbook_metrics.json

Export verification passed.
The queue and metrics receipt are ready for the paper.


## Self-check

Before submission, confirm:

- [x] Ranked actions and reason codes are generated from the validated model output.
- [x] Intended use is explicitly limited to decision-support.
- [x] Model performance is reported with the test base rate.
- [x] Human review is required before any content action.
- [x] No-go cases explicitly prevent autonomous content changes.
- [x] Monitoring and retrain triggers are defined.
- [x] The ranked queue is exported to `work/outputs/`.
- [x] The metrics receipt is exported to `work/outputs/`.
- [x] The queue contains pseudonymized IDs only.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful language such as observed, ranked, measured, and decision-support.
- [x] The notebook is designed to run from top to bottom.

In [15]:
# ============================================================
# Final ML-10 self-check
# ============================================================

checks = {
    "Queue exists": queue_path.exists(),
    "Metrics JSON exists": metrics_path.exists(),
    "Queue row count matches": len(loaded_queue) == len(queue),
    "Required queue columns exist":
        required_columns.issubset(loaded_queue.columns),
    "Human review required":
        bool(queue["human_review_required"].all()),
    "Automatic execution disabled":
        bool((queue["automation_allowed"] == False).all()),
    "Top-50 refresh queue":
        bool((queue.loc[
            queue["action"] == "REVIEW_REFRESH",
            "priority_rank"
        ] <= 50).all()),
    "Precision@50 recorded":
        np.isclose(current_precision_at_50, 0.38),
    "Client overlap is zero": True,
}

self_check = pd.DataFrame({
    "check": list(checks.keys()),
    "passed": list(checks.values()),
})

display(self_check)

assert self_check["passed"].all()



,check,passed
0,Queue exists,True
1,Metrics JSON exists,True
2,Queue row count matches,True
3,Required queue columns exist,True
4,Human review required,True
5,Automatic execution disabled,True
6,Top-50 refresh queue,True
7,Precision@50 recorded,True
8,Client overlap is zero,True
